# Expense Tracker — Kaggle Master Training

**Run All is enough for the standard training pipeline.** The notebook bootstraps the repository, installs a Kaggle-compatible scientific stack, installs the ML package, configures the runtime, fetches the configured Hugging Face datasets through the master entrypoint, trains the models, evaluates them, and leaves reports/graphs/models under `/kaggle/working`.

The Hugging Face token is read from the Kaggle environment as `HF_TOKEN` when needed; no token is stored in this notebook.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path('/kaggle/working/expense-tracker')
BRANCH = 'feature/ml-expense-intelligence'
REMOTE = 'https://github.com/Yoge-2004/expense-tracker.git'

# Kaggle sessions may restore /kaggle/working between runs. Always ensure the
# requested branch is present and up to date instead of reusing stale code.
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REMOTE, str(REPO)], check=True)

ML = REPO / 'ml'
SRC = ML / 'src'

# Kaggle ships a large preinstalled environment. Keep the project's scientific
# stack internally consistent instead of allowing pip to select incompatible
# NumPy/SciPy/sklearn combinations from the broad pyproject ranges. These pins
# remain inside the project's declared dependency ranges and avoid the known
# NumPy 2.5.x / SciPy / sklearn import failure seen in the Kaggle runtime.
SCIENTIFIC_PINS = [
    'numpy==2.3.3',
    'scipy==1.16.2',
    'scikit-learn==1.7.2',
]
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall', *SCIENTIFIC_PINS],
    check=True,
)

# Install the local package after the scientific stack is pinned. Also add src
# explicitly so the current notebook kernel can import the package immediately
# even if its sys.path was created before pip finished the editable install.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ML)], check=True)
sys.path.insert(0, str(SRC))

import expense_ml
print('ML code:', ML)
print('expense_ml:', expense_ml.__file__)

In [ ]:
import json
from expense_ml.resources import configure_resources

os.environ.setdefault('EXPENSE_ML_CPU_THREADS', 'auto')
os.environ.setdefault('EXPENSE_ML_DATALOADER_WORKERS', '8')
os.environ.setdefault('EXPENSE_ML_BATCH_SIZE', '32')
os.environ.setdefault('EXPENSE_ML_EVAL_BATCH_SIZE', '64')
os.environ.setdefault('EXPENSE_ML_MIXED_PRECISION', 'auto')
os.environ.setdefault('EXPENSE_ML_MAX_MERCHANTS', '250000')
os.environ.setdefault('EXPENSE_ML_DUPLICATE_MAX_ROWS', '500000')
os.environ.setdefault('EXPENSE_ML_NORMALIZE_CHUNK_SIZE', '250000')
os.environ['EXPENSE_ML_OUTPUT'] = '/kaggle/working/expense-ml-runs'
os.environ['EXPENSE_ML_DATA_CACHE'] = '/kaggle/working/expense-ml-data'

resources = configure_resources()
print(json.dumps(resources, indent=2))

if os.environ.get('HF_TOKEN'):
    print('HF_TOKEN: available to the training process')
else:
    print('HF_TOKEN: not set; public datasets can still be used, but gated datasets may fail')

In [ ]:
# Preflight imports before spending GPU time on data download/training.
import datasets, pandas, numpy, sklearn, scipy, torch, transformers

print('datasets:', datasets.__version__)
print('pandas:', pandas.__version__)
print('numpy:', numpy.__version__)
print('scipy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Run the single master entrypoint. It handles dataset fetching/caching,
# normalization, training, evaluation, model export, and report generation.
# This avoids loading the configured datasets twice in the notebook.
%run /kaggle/working/expense-tracker/ml/kaggle_train.py

In [ ]:
from pathlib import Path
import json

runs = sorted(
    p for p in Path('/kaggle/working/expense-ml-runs').glob('*') if (p / 'manifest.json').exists()
)
if not runs:
    raise RuntimeError('No completed master-training run was found.')

latest = runs[-1]
manifest = json.loads((latest / 'manifest.json').read_text())
print('Latest run:', latest)
print(json.dumps(manifest['resources'], indent=2))
print(json.dumps(manifest['models'], indent=2))
print((latest / 'reports' / 'REPORT.md').read_text())